# Train the thermal CNN-LSTM on a Colab GPU

Runs `05_train_cnn_lstm.py` and `06_evaluate_test.py` on a free Colab GPU instead of a laptop CPU.

**Before you start**
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Upload your local `Data\sequences` folder (the one `04_build_sequences.py` made, with `train/`, `val/`, `test/` inside) to Google Drive as
   `My Drive/thermal_behaviour/sequences`.

Then run the cells top to bottom. The trained model and the test report are saved to `My Drive/thermal_behaviour/` as training goes, so a disconnect doesn't lose the best checkpoint.

In [ ]:
# 1. Check the GPU is on
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
assert gpus, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run this cell again."

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/thermal_behaviour'   # change if you used another folder
import os
for split in ['train', 'val', 'test']:
    d = os.path.join(DRIVE_DIR, 'sequences', split)
    files = sorted(os.listdir(d)) if os.path.isdir(d) else 'MISSING'
    print(split, '->', files)

In [ ]:
# 3. Get the scripts
!rm -rf /content/repo
!git clone -q -b claude/gracious-dirac-kdb1q7 https://github.com/abhinaya-thota/thermal-behavior-recognition.git /content/repo
!ls /content/repo

In [ ]:
# 4. Copy the sequences to the Colab machine's local disk (much faster to read than Drive)
#    and send models/ and outputs/ straight to Drive.
WORK = '/content/work'
!rm -rf {WORK}
!mkdir -p {WORK}/data
!cp -r "{DRIVE_DIR}/sequences" {WORK}/data/sequences
!du -sh {WORK}/data/sequences/*

for d in ['models', 'outputs']:
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)
    os.symlink(os.path.join(DRIVE_DIR, d), os.path.join(WORK, d))
print("models/ and outputs/ ->", DRIVE_DIR)

In [ ]:
# 5. Settings for the GPU: a T4 has 15 GB of memory, so batch 32 fits.
#    If you get RESOURCE_EXHAUSTED / OOM, set this to 16 and re-run this cell and the next.
BATCH_SIZE = 32

import re
src = open('/content/repo/05_train_cnn_lstm.py').read()
src = re.sub(r'^BATCH_SIZE = \d+', f'BATCH_SIZE = {BATCH_SIZE}', src, flags=re.M)
open('/content/repo/05_train_cnn_lstm.py', 'w').write(src)
!grep -n "^BATCH_SIZE" /content/repo/05_train_cnn_lstm.py

In [ ]:
# 6. Train (the best model is saved to Drive after every improving epoch)
%cd {WORK}
!python /content/repo/05_train_cnn_lstm.py

In [ ]:
# 7. Evaluate on the test set
%cd {WORK}
!python /content/repo/06_evaluate_test.py

In [ ]:
# 8. Show the results (also saved in My Drive/thermal_behaviour/outputs/)
print(open(f'{WORK}/outputs/test_report.txt').read())
from IPython.display import Image, display
for f in ['training_accuracy.png', 'confusion_matrix_test.png']:
    p = f'{WORK}/outputs/{f}'
    if os.path.exists(p):
        display(Image(p))